In [0]:
%pip install azure-eventhub

In [0]:
"""
================================================================================
  Event Hub Producer: Auto-Discover Parquet from GitHub + Fake Data Generator
================================================================================
  Flow:
    1. Hit GitHub API → auto-find ALL .parquet files in the repo/folder
    2. Download & stream each file row-by-row → push to Event Hub
    3. Once all files are done → generate fake telemetry data (infinite loop)

  Schema expected (both real and fake):
    Chassis_no | latitude | longitude | event_timestamp | speed
================================================================================
"""

import io
import time
import json
import random
import string
import logging
from datetime import datetime, timedelta

import requests
import pandas as pd
from azure.eventhub import EventHubProducerClient, EventData

# ── Configure logging ──────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger(__name__)

# ══════════════════════════════════════════════════════════════════════════════
#  ① CONFIGURATION  –  Edit everything in this block
# ══════════════════════════════════════════════════════════════════════════════

# Azure Event Hub
EVENT_HUB_CONNECTION_STR = "${EVENT_HUB_KEY}"
EVENT_HUB_NAME           = "telematics-car-stream"

# ── GitHub repo settings ───────────────────────────────────────────────────────
# Owner and repo name
GITHUB_OWNER  = "ShoaibAttanKhan5887"          # e.g. "microsoft"
GITHUB_REPO   = "Car-Insurance-Datasets"           # e.g. "nyc-taxi-data"
GITHUB_BRANCH = "main"             # branch name: "main", "master", etc.

# Folder inside the repo where parquet files live.
# Use "" (empty string) to search the entire repo recursively.
GITHUB_FOLDER = "data/telematics"             # e.g. "data/parquet" or "" for whole repo

# (Optional) Personal Access Token – needed for private repos or to avoid
# GitHub's 60 req/hr unauthenticated rate limit.
# Leave as None for public repos with light usage.
GITHUB_TOKEN  = None               # e.g. "ghp_xxxxxxxxxxxx"

# ── Column mapping ─────────────────────────────────────────────────────────────
# Rename your parquet columns to the canonical schema names if they differ.
# Set to {} to skip renaming (assumes columns already match).
COLUMN_RENAME = {
    # "your_col":  "Chassis_no",
    # "lat":       "latitude",
    # "lon":       "longitude",
    # "ts":        "event_timestamp",
    # "spd":       "speed",
}

# ── Timing ─────────────────────────────────────────────────────────────────────
ROW_DELAY_SECONDS  = 0.5   # delay between batches while replaying parquet data
FAKE_DELAY_SECONDS = 1.0   # delay between events during fake data generation

# ── Fake data bounding box (default: New York City area) ──────────────────────
FAKE_LAT_RANGE   = (40.4774,  40.9176)
FAKE_LON_RANGE   = (-74.2591, -73.7004)
FAKE_SPEED_RANGE = (0.0, 120.0)     # km/h or mph — your choice

# ── Batching ───────────────────────────────────────────────────────────────────
BATCH_SIZE = 10   # number of events per Event Hub send call

# ══════════════════════════════════════════════════════════════════════════════

# ── GitHub discovery ──────────────────────────────────────────────────────────

def _github_headers() -> dict:
    headers = {"Accept": "application/vnd.github+json"}
    if GITHUB_TOKEN:
        headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"
    return headers

def _discover_parquet_urls() -> list:
    """
    Use the GitHub Trees API (recursive) to find every .parquet file
    inside GITHUB_FOLDER (or the whole repo if GITHUB_FOLDER is empty).
    Returns a list of raw download URLs.
    """
    tree_url = (
        f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}"
        f"/git/trees/{GITHUB_BRANCH}?recursive=1"
    )
    log.info("🔍  Fetching repo tree from: %s", tree_url)

    resp = requests.get(tree_url, headers=_github_headers(), timeout=30)

    if resp.status_code == 401:
        raise RuntimeError("GitHub returned 401 – check your GITHUB_TOKEN.")
    if resp.status_code == 403:
        raise RuntimeError(
            "GitHub returned 403 – rate limit hit or private repo needs a token."
        )
    resp.raise_for_status()

    data = resp.json()
    truncated = data.get("truncated", False)
    if truncated:
        log.warning(
            "⚠️  GitHub tree response was truncated (repo too large). "
            "Some files may be missed. Consider narrowing GITHUB_FOLDER."
        )

    folder_prefix = GITHUB_FOLDER.strip("/") + "/" if GITHUB_FOLDER.strip("/") else ""

    raw_urls = []
    for item in data.get("tree", []):
        path = item.get("path", "")
        if item.get("type") == "blob" and path.endswith(".parquet"):
            if not folder_prefix or path.startswith(folder_prefix):
                raw_url = (
                    f"https://raw.githubusercontent.com/"
                    f"{GITHUB_OWNER}/{GITHUB_REPO}/{GITHUB_BRANCH}/{path}"
                )
                raw_urls.append(raw_url)
                log.info("    📄 Found: %s", path)

    log.info("✅  Discovered %d parquet file(s).", len(raw_urls))
    return raw_urls

# ── Helpers ────────────────────────────────────────────────────────────────────

def _random_chassis() -> str:
    """Generate a 17-character VIN-style chassis number."""
    chars = (
        string.ascii_uppercase.replace("I", "").replace("O", "").replace("Q", "")
        + string.digits
    )
    return "".join(random.choices(chars, k=17))

def _row_to_dict(row) -> dict:
    record = row.to_dict()
    for key in ("event_timestamp",):
        if key in record and not isinstance(record[key], str):
            record[key] = str(record[key])
    return record

def _send_batch(producer: EventHubProducerClient, records: list) -> None:
    event_batch = producer.create_batch()
    for rec in records:
        event_batch.add(EventData(json.dumps(rec, default=str).encode("utf-8")))
    producer.send_batch(event_batch)

# ── Stage 1: Replay Parquet files ─────────────────────────────────────────────

def stream_parquet_files(producer: EventHubProducerClient, urls: list) -> None:
    if not urls:
        log.warning("No parquet files found – skipping replay stage.")
        return

    for file_num, url in enumerate(urls, start=1):
        log.info("📥  [%d/%d] Downloading: %s", file_num, len(urls), url)
        try:
            resp = requests.get(url, headers=_github_headers(), timeout=60)
            resp.raise_for_status()
        except Exception as exc:
            log.error("    ❌ Failed – %s", exc)
            continue

        df = pd.read_parquet(io.BytesIO(resp.content))
        log.info("    Loaded %d rows | columns: %s", len(df), list(df.columns))

        if COLUMN_RENAME:
            df = df.rename(columns=COLUMN_RENAME)

        required = {"chassis_no", "latitude", "longitude", "event_timestamp", "speed"}
        missing  = required - set(df.columns)
        if missing:
            log.warning("    ⚠️  Skipping – missing columns: %s", missing)
            continue

        buffer = []
        for idx, row in df.iterrows():
            buffer.append(_row_to_dict(row))
            if len(buffer) >= BATCH_SIZE:
                _send_batch(producer, buffer)
                log.info("    ✅ Sent batch of %d (row index: %s)", len(buffer), idx)
                buffer.clear()
                time.sleep(ROW_DELAY_SECONDS)

        if buffer:
            _send_batch(producer, buffer)
            log.info("    ✅ Sent final batch of %d", len(buffer))

        log.info("    ✔  File %d/%d done.", file_num, len(urls))

    log.info("🏁  All parquet files replayed successfully.")

# ── Stage 2: Fake real-time data generator ────────────────────────────────────

def generate_fake_data(producer: EventHubProducerClient) -> None:
    log.info("🤖  Switching to fake data generation  (Ctrl+C to stop) …")
    current_time = datetime.utcnow()
    buffer = []

    while True:
        record = {
            "Chassis_no":      _random_chassis(),
            "latitude":        round(random.uniform(*FAKE_LAT_RANGE),  6),
            "longitude":       round(random.uniform(*FAKE_LON_RANGE),  6),
            "event_timestamp": current_time.strftime("%Y-%m-%d %H:%M:%S"),
            "speed":           round(random.uniform(*FAKE_SPEED_RANGE), 2),
        }
        buffer.append(record)
        log.info(
            "  ➤ %s  lat=%-10s  lon=%-11s  ts=%s  speed=%s",
            record["Chassis_no"], record["latitude"], record["longitude"],
            record["event_timestamp"], record["speed"],
        )

        if len(buffer) >= BATCH_SIZE:
            _send_batch(producer, buffer)
            buffer.clear()

        current_time += timedelta(seconds=FAKE_DELAY_SECONDS)
        time.sleep(FAKE_DELAY_SECONDS)

# ── Entry point ───────────────────────────────────────────────────────────────

def main() -> None:
    log.info("🚀  Starting Event Hub Producer")

    # Step 1 – auto-discover all parquet files from GitHub
    parquet_urls = _discover_parquet_urls()

    # Step 2 – create producer and stream
    producer = EventHubProducerClient.from_connection_string(
        conn_str=EVENT_HUB_CONNECTION_STR,
        eventhub_name=EVENT_HUB_NAME,
    )
    try:
        with producer:
            stream_parquet_files(producer, parquet_urls)   # Stage 1: replay
            generate_fake_data(producer)                   # Stage 2: fake (infinite)
    except KeyboardInterrupt:
        log.info("⛔  Interrupted by user. Exiting.")

if __name__ == "__main__":
    main()